# dem4cli version 2 demo

## Description 


## Functions

**Part 1. Demographics preprocesing**

- load_cohort_sizes() loads cohort sizes from Wittgenstein SSP IIASA reconstructions and projections
    - v1 is from WCDEv2, v2 is from WCDE v3.2-beta
- interpolate_cohortsize_countries() interpolates from 5-year age brackets to exact ages
- load_population() loads gridded pop data
    - v1 from ISIMIP3, v2 from COMPASS (pre-CMIP7 version)
- load_unwpp_lifeexpectancy() loads and cleans life expectancy data e(x) at age 5 from UNWPP2024
- get_life_expectancies() goes from raw e(x) data to period -> cohort life expectancy at birth 
- load_countrymasks_fillcoasts() loads fractional countrymasks, with options for filling coastal pixels to sum to 1 

**Part 2. Lifetime exposure**
- GMT mapping 
- lifetime exposure computation


## Notes
- load_country_metadata() only v1 
- load_country_stats() only v1 and not very useful
- get_gridscale_demographics() and population_demographics_gridscale_global() only v1
- 197 countries included in all data sources and with SSP cohort size projections (218 included in all 3 but without cohort size projs)
    - Could get cohort sizes from UNWPP2024! Might lose less countries


## To Do
- wrapper fxn in same(ish) format as S2S


In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import geopandas as gpd
import pickle as pk
from scipy import interpolate
#import regionmask
import glob, os, re, sys
import openpyxl
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 80)
%matplotlib inline 

sys.path.append('..')
from population_demographics_v2 import * 

In [2]:
flags

# in init function user can define these 

{'version': 2, 'pop_resolution': 0.1, 'GMT_mapping': 'year_to_year'}

In [3]:
bbox_europe = [ 31.99,  71.09, -14.96,  34.94]

## Part 1. Demographics preprocessing

In [3]:
df_cohort_sizes, ages, years = load_cohort_sizes(dir_cohortsizes, ssp=2, by_sex=False)

# loads raw cohort size data from historical + ssp and cleans to keep only relevant information

# TODO: note that could get this from UNWPP2024! 

In [4]:
da_cohort_size = interpolate_cohortsize_countries(
                        df_cohort_sizes,
                        ages,
                        years,
                    )

# interpolates cohort sizes from 5 year to single year and corrects to preserve mean

# TODO: with this new data the neg numbers happens more, check if this is big or small issue

interpolating cohort sizes per country
after interpolation and mean-preserving correction there are some neg numbers in 6, Anguilla, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 15, Bahrain, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 29, British Virgin Islands, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 32, Burkina Faso, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 39, Central African Republic, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 61, El Salvador, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 65, Eswatini, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 68, Faroe Islands, setting them t

In [ ]:
df_unwpp = load_unwpp_lifeexpectancy(filepath_lifeexpectancy = filepath_lifeexpectancy) # load life expectancy data and clean 
df_life_expectancy_5 = get_life_expectancies(df_unwpp,
                                            #extend=True,
                                            start_birthyear=1950,
                                            end_birthyear=2025) # go from 'period' to 'cohort' life expectancy

In [ ]:
da_population = load_population(
    dir_population= dir_population, 
    startyear=1950,
    endyear=2100,
    ssp=2,
    urbanrural=False,
    bbox = [ 31.99,  71.09, -14.96,  34.94] # optional : latmin, latmax, lonmin, lonmax

)

# load gridded population data, crop over Europe (optional cropping)


opening compass - historical
opening compass - ssp2


In [4]:
# open country masks

da_countrymasks = load_countrymasks_fillcoasts(
                            filepath_countrymask,
                            preprocess=False, # True if you want to preprocess
                            fillcoast=False, # fill coastal pixels to not lose coastal pops (done in preprocessed files)
                            fix_smallislands=False, # done in preprocessed input files for 0.5, not for 0.1 - TODO: check if necessary at 0.1 or not ! 
                            bbox=bbox_europe,
                            )

# these are already preprocessed (cleaned country names and coastal pixels are filled to sum to 1 to not lose population)




In [8]:
# lookup table

# fxn that matches the names in cohort size and this and mask / lookup table for v2 


#TODO just add column from cohort size name to Dominik's lookup table matching based on ISO number and save 

df_lookuptable = pd.read_csv(filepath_lookuptable) # new



In [4]:
# wrapper function that runs everything for S2S / lifetime exposure calcs


d_countries = preprocess_all_country_data(

    dir_cohortsizes = dir_cohortsizes,  # cohort size data
    ssp=2, 
    by_sex=False,                       # NOTE by_sex not implemented
                                            
    filepath_lifeexpectancy = filepath_lifeexpectancy, # life expectancy data
    start_birthyear=1950,
    end_birthyear=2025, 

    dir_population= dir_population,     # gridded pop data 
    startyear=1950,
    endyear=2100,
    urbanrural=False,                   # NOTE urbanrural not implemented for v2
    bbox = bbox_europe,

    filepath_countrymask = filepath_countrymask ,   # country masks 
    preprocess=False,                               # NOTE preprocessing is already done in standard input files 
    fillcoast=False, 
    fix_smallislands=False,
)




#TODO filter countries to have only the ones included in all the 



interpolating cohort sizes per country
after interpolation and mean-preserving correction there are some neg numbers in 6, Anguilla, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 15, Bahrain, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 29, British Virgin Islands, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 32, Burkina Faso, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 39, Central African Republic, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 61, El Salvador, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 65, Eswatini, setting them to zero
after interpolation and mean-preserving correction there are some neg numbers in 68, Faroe Islands, setting them t

In [5]:
d_countries

{'info_pop': None,
 'borders': <xarray.DataArray (lat: 391, lon: 499, country: 49)>
 dask.array<getitem, shape=(391, 499, 49), dtype=float32, chunksize=(337, 430, 18), chunktype=numpy.ndarray>
 Coordinates:
   * lon      (lon) float64 -14.95 -14.85 -14.75 -14.65 ... 34.65 34.75 34.85
   * lat      (lat) float64 32.05 32.15 32.25 32.35 ... 70.75 70.85 70.95 71.05
   * country  (country) object 'ALB' 'AND' 'AUT' 'BEL' ... 'TUN' 'TUR' 'UKR',
 'population_map': <xarray.DataArray 'total-population' (time: 151, lat: 391, lon: 499)>
 dask.array<concatenate, shape=(151, 391, 499), dtype=float32, chunksize=(1, 180, 349), chunktype=numpy.ndarray>
 Coordinates:
   * lon      (lon) float64 -14.95 -14.85 -14.75 -14.65 ... 34.65 34.75 34.85
   * lat      (lat) float64 32.05 32.15 32.25 32.35 ... 70.75 70.85 70.95 71.05
   * time     (time) int64 1950 1951 1952 1953 1954 ... 2096 2097 2098 2099 2100
 Attributes:
     units:      1
     long_name:  Population_count,
 'birth_years': None,
 'life_expect

### Check data

In [ ]:
da_cohort_size # from ssps 

# NOTE! overlap between mask, UNWPP and SSP projections is 218 countries 

# NOTE! of these only 197 countries have full cohort size projections into future!

# TODO could check if with UNWPP cohort size projections this is better - but then would be inconsistent with gridded pop national totals (that come from SSPs) - decide how to deal with this 

<xarray.DataArray 'cohort_size' (country: 236, time: 151, ages: 105)>
array([[[2.73567360e-01, 2.61743480e-01, 2.49919600e-01, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [2.73309184e-01, 2.62139952e-01, 2.50970720e-01, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [2.73051008e-01, 2.62536424e-01, 2.52021840e-01, ...,
         0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        ...,
        [1.19588425e+00, 1.20333086e+00, 1.21077748e+00, ...,
         2.38331824e-04, 0.00000000e+00, 0.00000000e+00],
        [1.18756014e+00, 1.19529399e+00, 1.20302784e+00, ...,
         2.56643380e-04, 0.00000000e+00, 0.00000000e+00],
        [1.17923604e+00, 1.18725712e+00, 1.19527820e+00, ...,
         2.74954937e-04, 0.00000000e+00, 0.00000000e+00]],

       [[3.90316400e-02, 3.79113200e-02, 3.67910000e-02, ...,
         1.99376947e-06, 0.00000000e+00, 0.00000000e+00],
        [4.13674160e-02, 4.00184880e-02, 3.86695600e-02, ...,
         2.18182113e-06, 0.00000000e+00, 0.00000000e+00],
        [4.37031920e-02, 4.21256560e-02, 4.05481200e-02, ...,
         2.36987279e-06, 0.00000000e+00, 0.00000000e+00],
...
        [5.17906600e-01, 5.20906640e-01, 5.23906680e-01, ...,
         2.24356300e-04, 0.00000000e+00, 0.00000000e+00],
        [5.14445400e-01, 5.17453120e-01, 5.20460840e-01, ...,
         2.36712206e-04, 0.00000000e+00, 0.00000000e+00],
        [5.10984200e-01, 5.13999600e-01, 5.17015000e-01, ...,
         2.49068113e-04, 0.00000000e+00, 0.00000000e+00]],

       [[1.02697080e-01, 9.89636400e-02, 9.52302000e-02, ...,
         3.92156863e-07, 0.00000000e+00, 0.00000000e+00],
        [1.10939600e-01, 1.06367600e-01, 1.01795600e-01, ...,
         3.55979011e-07, 0.00000000e+00, 0.00000000e+00],
        [1.19182120e-01, 1.13771560e-01, 1.08361000e-01, ...,
         3.19801160e-07, 0.00000000e+00, 0.00000000e+00],
        ...,
        [2.54660296e-01, 2.56762928e-01, 2.58865560e-01, ...,
         4.76045919e-04, 0.00000000e+00, 0.00000000e+00],
        [2.52330008e-01, 2.54450744e-01, 2.56571480e-01, ...,
         5.10386444e-04, 0.00000000e+00, 0.00000000e+00],
        [2.49999720e-01, 2.52138560e-01, 2.54277400e-01, ...,
         5.44726970e-04, 0.00000000e+00, 0.00000000e+00]]])
Coordinates:
  * country  (country) object 'Afghanistan' 'Albania' ... 'Zambia' 'Zimbabwe'
  * time     (time) int64 1950 1951 1952 1953 1954 ... 2096 2097 2098 2099 2100
  * ages     (ages) int64 0 1 2 3 4 5 6 7 8 ... 96 97 98 99 100 101 102 103 104

In [8]:
da_population

<xarray.DataArray 'total-population' (time: 151, lat: 391, lon: 499)>
dask.array<concatenate, shape=(151, 391, 499), dtype=float32, chunksize=(1, 180, 349), chunktype=numpy.ndarray>
Coordinates:
  * lon      (lon) float64 -14.95 -14.85 -14.75 -14.65 ... 34.65 34.75 34.85
  * lat      (lat) float64 32.05 32.15 32.25 32.35 ... 70.75 70.85 70.95 71.05
  * time     (time) int64 1950 1951 1952 1953 1954 ... 2096 2097 2098 2099 2100
Attributes:
    units:      1
    long_name:  Population_count

In [ ]:
df_life_expectancy_5

# life expectancy at age 5 (corrected to be at birth and period) for each country/birth year - last data is for 2018 birth year, extra years are held constant

# TODO: country names from UNWPP life expectancy - to do fxn that matches the names in cohort size and this and mask / lookup table for v2 

Country,Afghanistan,Albania,Algeria,American Samoa,Andorra,Angola,Anguilla,Antigua and Barbuda,Argentina,Armenia,...,Uruguay,Uzbekistan,Vanuatu,Venezuela (Bolivarian Republic of),Viet Nam,Wallis and Futuna Islands,Western Sahara,Yemen,Zambia,Zimbabwe
Year,,,,,,,,,,,,,,,,,,,,,
1945,53.3261,69.2296,63.5840,72.0589,76.7722,59.7620,69.7683,71.8547,73.1144,72.5277,...,76.2805,70.9224,60.4022,66.0496,62.9989,59.1348,57.6815,62.4295,67.3650,66.3182
1946,53.5996,69.4998,64.1647,72.1837,77.0034,59.8569,70.0189,72.1268,73.4322,72.7482,...,76.4038,71.0670,58.9680,66.7221,63.3894,59.2239,57.9052,62.5490,67.5156,66.5690
1947,53.8688,70.1581,64.1422,72.4278,77.2782,59.9553,70.2406,72.3293,74.1116,72.9654,...,76.5524,71.1997,61.3336,67.2419,63.7909,59.3332,58.1262,62.6602,67.7378,66.8106
1948,54.1415,70.9094,64.0927,72.6784,77.7080,60.0900,70.4861,72.5323,73.9636,73.1779,...,76.6936,71.3296,61.6748,67.7611,64.1265,59.5364,58.3487,62.8188,67.9695,67.0438
1949,54.1671,71.5560,59.9824,72.8803,78.1366,60.1895,70.7363,72.7200,74.4988,73.3851,...,76.8463,71.4600,62.1492,68.2673,66.4409,59.7734,58.5814,62.9305,68.1568,67.2780
1950,54.7051,72.2933,60.0833,73.0935,78.4736,60.3326,71.0196,72.9259,73.9860,73.5868,...,76.9994,71.5994,62.5983,68.7405,68.7827,60.0066,58.8167,63.0338,68.3711,67.5054
1951,54.9914,73.1170,59.9968,73.3142,78.7848,60.4601,71.3453,73.1271,74.8442,73.7835,...,77.1543,71.7428,63.0074,69.2576,69.2194,60.2678,59.0545,63.2106,68.5591,67.7423
1952,55.3287,74.0791,59.8907,73.5873,79.0685,60.5222,71.6498,73.3441,74.3825,73.9763,...,77.3030,71.8910,63.3789,69.7610,69.5801,60.5338,59.2911,63.3482,68.7140,67.9825
1953,55.6465,75.0774,59.7686,74.0095,79.4490,60.5925,71.9511,73.5912,75.4063,74.1662,...,77.4524,72.0515,63.7587,70.2772,69.9530,60.8083,59.5263,63.5801,68.9539,68.2102


In [5]:
da_countrymasks

<xarray.DataArray (lat: 391, lon: 499, country: 49)>
dask.array<getitem, shape=(391, 499, 49), dtype=float32, chunksize=(337, 430, 18), chunktype=numpy.ndarray>
Coordinates:
  * lon      (lon) float64 -14.95 -14.85 -14.75 -14.65 ... 34.65 34.75 34.85
  * lat      (lat) float64 32.05 32.15 32.25 32.35 ... 70.75 70.85 70.95 71.05
  * country  (country) object 'ALB' 'AND' 'AUT' 'BEL' ... 'TUN' 'TUR' 'UKR'

In [ ]:
da_countrymasks.country


# TODO: try to plot this - match ISO3 to numeric ISO3 

<xarray.DataArray 'country' (country: 49)>
array(['ALB', 'AND', 'AUT', 'BEL', 'BGR', 'BIH', 'BLR', 'CHE', 'CYP', 'CZE',
       'DEU', 'DNK', 'DZA', 'ESP', 'EST', 'FIN', 'FRA', 'FRO', 'GBR', 'GRC',
       'HRV', 'HUN', 'IMN', 'IRL', 'ISL', 'ISR', 'ITA', 'LBY', 'LTU', 'LUX',
       'LVA', 'MAR', 'MDA', 'MKD', 'MLT', 'MNE', 'NLD', 'NOR', 'POL', 'PRT',
       'ROU', 'RUS', 'SRB', 'SVK', 'SVN', 'SWE', 'TUN', 'TUR', 'UKR'],
      dtype=object)
Coordinates:
  * country  (country) object 'ALB' 'AND' 'AUT' 'BEL' ... 'TUN' 'TUR' 'UKR'

In [ ]:
df_lookuptable

,SSP name,ISO name,WPP name,ISO alpha-2,ISO alpha-3,ISO numeric,Data availability,Suggested gapfill
0,Afghanistan,Afghanistan,Afghanistan,AF,AFG,4,Full historical + SSP,NaN
1,Albania,Albania,Albania,AL,ALB,8,Full historical + SSP,NaN
2,Algeria,Algeria,Algeria,DZ,DZA,12,Full historical + SSP,NaN
3,Angola,Angola,Angola,AO,AGO,24,Full historical + SSP,NaN
4,Antigua and Barbuda,Antigua and Barbuda,Antigua and Barbuda,AG,ATG,28,Full historical + SSP,NaN
...,...,...,...,...,...,...,...,...
244,NaN,Norfolk Island,NaN,NF,NFK,574,No data,Rest of Asia (R10)
245,NaN,Pitcairn,NaN,PN,PCN,612,No data,Rest of Asia (R10)
246,NaN,South Georgia and the South Sandwich Islands,NaN,GS,SGS,239,No population,NaN
247,NaN,Svalbard and Jan Mayen,NaN,SJ,SJM,744,No data,Europe (R10)


In [ ]:
df_lookuptable[df_lookuptable['ISO alpha-3'].isin(da_countrymasks.country.values)].reset_index(drop=True)


# all the countries included in bounding box - decide whether to get rid of some!


,SSP name,ISO name,WPP name,ISO alpha-2,ISO alpha-3,ISO numeric,Data availability,Suggested gapfill
0,Albania,Albania,Albania,AL,ALB,8,Full historical + SSP,NaN
1,Algeria,Algeria,Algeria,DZ,DZA,12,Full historical + SSP,NaN
2,Austria,Austria,Austria,AT,AUT,40,Full historical + SSP,NaN
3,Belarus,Belarus,Belarus,BY,BLR,112,Full historical + SSP,NaN
4,Belgium,Belgium,Belgium,BE,BEL,56,Full historical + SSP,NaN
5,Bosnia and Herzegovina,Bosnia and Herzegovina,Bosnia and Herzegovina,BA,BIH,70,Full historical + SSP,NaN
6,Bulgaria,Bulgaria,Bulgaria,BG,BGR,100,Full historical + SSP,NaN
7,Croatia,Croatia,Croatia,HR,HRV,191,Full historical + SSP,NaN
8,Cyprus,Cyprus,Cyprus,CY,CYP,196,Full historical + SSP,NaN
9,Czechia,Czechia,Czechia,CZ,CZE,203,Full historical + SSP,NaN


## Part 2. Lifetime exposure

In [ ]:
# GMT-mapping + lifetime exposure functions

# TODO 


